# Chương 6 – Full Supervised Fine-Tuning (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/phuongtnd1994/ptnd-ModelAdaptationBook/blob/main/code/chapter06/chapter06_colab.ipynb)

Full SFT cập nhật **toàn bộ** 4B tham số → cần ~32,5 GB VRAM để train. Notebook có 2 đường:

- **Đường A (L4 24 GB trở lên):** tải model SFT đã train sẵn từ Hugging Face rồi chạy inference,
  đánh giá, behavioral tests, safety regression.
- **Đường B (A100 40 GB / H100):** tự train full SFT. Cần ~80 GB ổ đĩa trống cho checkpoint.

## 0. Cài đặt môi trường

Chạy lần lượt các ô dưới đây **một lần** mỗi khi mở runtime mới. Ô đầu tiên clone repo, cài package của sách và tạo hàm `run()` để chạy lệnh và in log trực tiếp ra notebook.

> Nếu pip báo `ERROR: pip's dependency resolver ...` về `gradio`, `diffusers`, `huggingface-hub` hoặc `safetensors`: **bỏ qua được**. Đó là các thư viện Colab cài sẵn mà sách không dùng; sách cố ý giữ `transformers<5` và `safetensors<0.6`.

In [1]:
import os, re, sys, subprocess

# Repo chứa code của sách. Nếu fork của bạn để private, đổi sang repo gốc:
#   https://github.com/bahree/ModelAdaptationBook.git
REPO_URL = "https://github.com/phuongtnd1994/ptnd-ModelAdaptationBook.git"
REPO_DIR = "/content/ModelAdaptationBook"
CODE_DIR = f"{REPO_DIR}/code"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

os.environ["PYTHONUNBUFFERED"] = "1"   # log của script con hiện ra ngay
os.environ["WANDB_DISABLED"] = "true"  # không dùng Weights & Biases trên Colab
# Sách chỉ dùng PyTorch. Không cho transformers nạp TensorFlow có sẵn trên Colab
# (tránh cảnh báo giả "Could not find cuda drivers" của TensorFlow).
os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


def run(cmd: str) -> None:
    """Chạy một lệnh shell từ thư mục code/, in log trực tiếp, báo lỗi nếu thất bại."""
    # Một số script không tự tạo thư mục output -> tạo sẵn trước khi chạy.
    for flag, path in re.findall(r"--(output_dir|output|out)\s+[\"']?([^\s\"']+)", cmd):
        target = path if flag == "output_dir" or not os.path.splitext(path)[1] else os.path.dirname(path)
        if target:
            os.makedirs(os.path.join(CODE_DIR, target), exist_ok=True)
    print(f"$ {cmd}\n", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=CODE_DIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh thất bại (exit code {p.returncode}): {cmd}")


run('pip install -q -e ".[dev]"')
print("\nĐã cài xong. Thư mục làm việc:", os.getcwd())

$ pip install -q -e ".[dev]"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires safetensors>=0.8.0, but you have safetensors 0.5.3 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.

Đã cài xong. Thư mục làm việc: /content/ModelAdaptationBook/code


In [3]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} | {mem:.1f} GB | compute capability {cap[0]}.{cap[1]}")
    if cap < (8, 0):
        print("CẢNH BÁO: GPU này (ví dụ T4) không hỗ trợ bf16 gốc. Các bước TRAINING "
              "trong sách dùng bf16 -> hãy đổi sang L4 hoặc A100 "
              "(Runtime > Change runtime type).")
else:
    print("Không có GPU. Chỉ các bước chạy trên CPU / gọi API hoạt động được.")

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition | 102.0 GB | compute capability 12.0


### Khoá API (Colab Secrets)

Mở biểu tượng 🔑 (Secrets) ở thanh bên trái, thêm:

- `HF_TOKEN` = token Hugging Face (không bắt buộc)

và bật *Notebook access*. Nếu chưa thêm, ô dưới sẽ hỏi bạn nhập tay.

In [4]:
import os
from getpass import getpass


def get_secret(name, required=False):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    if required:
        return getpass(f"Nhập {name}: ")
    return None


hf = get_secret("HF_TOKEN")  # không bắt buộc, giúp tải model nhanh hơn
if hf:
    os.environ["HF_TOKEN"] = hf
print("Đã nạp khoá:", [k for k in ("ANTHROPIC_API_KEY", "HF_TOKEN") if os.environ.get(k)])

Đã nạp khoá: ['HF_TOKEN']


### Tải model đã train sẵn từ Hugging Face

Tác giả sách đã publish model của từng chương tại [`bahree/ModelAdaptationBook`](https://huggingface.co/bahree/ModelAdaptationBook). Ô dưới tải đúng các thư mục cần cho chương này về `/content/hub/` để các script dùng như đường dẫn local (không cần tự train các chương trước).

In [5]:
from huggingface_hub import snapshot_download

HUB_DIR = "/content/hub"
SUBFOLDERS = ["ch6-sft"]
snapshot_download("bahree/ModelAdaptationBook", local_dir=HUB_DIR,
                  allow_patterns=[f"{s}/*" for s in SUBFOLDERS])
for s in SUBFOLDERS:
    print(s, "->", f"{HUB_DIR}/{s}", sorted(os.listdir(f"{HUB_DIR}/{s}"))[:6])

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

eval_report.json:   0%|          | 0.00/825 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/499 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

ch6-sft -> /content/hub/ch6-sft ['README.md', 'added_tokens.json', 'chat_template.jinja', 'config.json', 'eval_report.json', 'generation_config.json']


## Đường A – Dùng model SFT đã publish

`SFT_DIR` trỏ tới model tải ở bước 0. Nếu bạn train ở Đường B, đổi thành `chapter06/runs/sft_run1`.

In [6]:
SFT_DIR = "/content/hub/ch6-sft"

### A1. Sinh câu trả lời (Listing 6.3)

In [7]:
run(f'python -m chapter06.generate --model_dir {SFT_DIR} --prompt "How do I troubleshoot a VPN connection failure?"')

$ python -m chapter06.generate --model_dir /content/hub/ch6-sft --prompt "How do I troubleshoot a VPN connection failure?"

Loading model from /content/hub/ch6-sft
The tokenizer you are loading from '/content/hub/ch6-sft' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.80it/s]
The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

Prompt:   How do I troubleshoot a VPN connection failure?
Response: **Summary:** Check the VPN client logs, verify the server's IP address, and ensure the server is reachable from your network.

**Steps:**
1. Check the VPN client logs 

### A2. Đánh giá base vs SFT trên test split (Listing 6.5)

In [ ]:
run(f"python -m chapter06.eval_sft --data_dir data/it_support --split test "
    f"--model_dir {SFT_DIR} --output chapter06/runs/colab_eval/eval_report.json")

$ python -m chapter06.eval_sft --data_dir data/it_support --split test --model_dir /content/hub/ch6-sft --output chapter06/runs/colab_eval/eval_report.json

Loaded 50 test examples from data/it_support/test.jsonl

--- Evaluating base model ---
`torch_dtype` is deprecated! Use `dtype` instead!

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]


In [ ]:
import json, pathlib
p = pathlib.Path("chapter06/runs/colab_eval/eval_report.json")
print(json.dumps(json.loads(p.read_text()), indent=2, ensure_ascii=False)[:3000] if p.exists() else f"Chưa có {p}")

### A3. Behavioral tests (Listing 6.6)

In [ ]:
run(f"python -m chapter06.scripts.behavioral_tests --model_dir {SFT_DIR} --also_test_base")

### A4. Safety regression suite (Listing 6.9)

In [ ]:
run(f"python -m chapter06.scripts.safety_regression --model_dir {SFT_DIR} --output_dir chapter06/eval/safety_colab")

## Đường B – Tự train full SFT (cần A100 40 GB+)

Ô đầu kiểm tra VRAM; ô sau chạy smoke test 2 bước rồi train đầy đủ (~45–60 phút trên A100).

In [ ]:
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f"VRAM: {vram:.0f} GB")
assert vram >= 39, "Full SFT cần >= 40 GB VRAM (A100/H100). Hãy dùng Đường A."
!df -h /content | tail -1

In [ ]:
run("python -m chapter06.train_sft --train data/it_support_fmt/train.jsonl "
    "--valid data/it_support_fmt/valid.jsonl --out chapter06/runs/sft_smoke --max_steps 2 --report_to none")
run("rm -rf chapter06/runs/sft_smoke")

In [ ]:
run("python -m chapter06.train_sft --train data/it_support_fmt/train.jsonl "
    "--valid data/it_support_fmt/valid.jsonl --out chapter06/runs/sft_run1 --report_to none")

### B2. Theo dõi quá trình train (Listing 6.4)

Đọc `trainer_state.json` của checkpoint mới nhất.

In [ ]:
import glob, re
ckpts = sorted(glob.glob("chapter06/runs/sft_run1/checkpoint-*"), key=lambda p: int(re.findall(r"\d+", p)[-1]))
run(f"python -m chapter06.scripts.monitor {ckpts[-1]}") if ckpts else print("Chưa có checkpoint")

Sau khi train xong, đặt `SFT_DIR = "chapter06/runs/sft_run1"` và chạy lại các ô A1–A4.

## Lưu kết quả về Google Drive (tuỳ chọn)

Runtime Colab sẽ bị xoá khi ngắt kết nối. Chạy ô dưới để copy thư mục kết quả sang Drive.

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/ModelAdaptationBook"
for p in ["chapter06/runs/colab_eval", "chapter06/eval/safety_colab"]:
    src = os.path.join(CODE_DIR, p)
    dst = os.path.join(DEST, p)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("Đã lưu", p, "->", dst)
    elif os.path.isfile(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        print("Đã lưu", p, "->", dst)